# 2 · Safety Stock and Replenishment Policy

<a href="https://colab.research.google.com/github/FH-Prevail/jku_course/blob/main/notebooks/2_forecast_to_stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

If the button does not open, use this link, or copy it into the browser: <a href="https://colab.research.google.com/github/FH-Prevail/jku_course/blob/main/notebooks/2_forecast_to_stock.ipynb" target="_parent">https://colab.research.google.com/github/FH-Prevail/jku_course/blob/main/notebooks/2_forecast_to_stock.ipynb</a>

Or in Colab: **File, Open notebook, GitHub**, type `FH-Prevail/jku_course` and choose the notebook.

*UE Digital Analytics im Handel · JKU Linz · 03.10.2026 · Sina Mirshahi, Logistikum*

Choose **Copy to Drive** (File, Save a copy in Drive), then **Runtime → Run all**. Work in pairs. You do not write code: read the tables and use the controls.

The first cell fetches the course data, the precomputed forecasts and the display helpers from the course repository, github.com/FH-Prevail/jku_course. It does not train a model, and it takes a few seconds. The cell can stay collapsed.

**If a control is missing:** use the static example directly above it. If you edited a cell by accident, Undo, then run it again; otherwise reopen a fresh copy from the repository. Restarting does not undo edits.

**Optional Gemini help:** select an output and ask “Explain this table in plain English. Separate what it shows from what it does not prove.” Check the answer against the numbers. If Gemini is unavailable, ask your partner or lecturer. Use only the synthetic course data.


In [1]:
#@title Start here: run this cell once. It fetches the course data from GitHub. Nothing to edit.
import os, sys, pathlib, shutil, urllib.request, importlib, importlib.util, subprocess
required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "scipy": "scipy", "ipywidgets": "ipywidgets"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
SOURCE = os.environ.get("T2_SOURCE") or "https://raw.githubusercontent.com/FH-Prevail/jku_course/main"
FILES = ['src/classroom.py', 'data/classroom/results.json', 'data/classroom/products.csv', 'data/classroom/demand.csv', 'data/classroom/comparisons.csv', 'data/classroom/inventory.csv']
folder = pathlib.Path("t2_classroom")
folder.mkdir(exist_ok=True)
try:
    for f in FILES:
        target = folder / pathlib.Path(f).name
        if SOURCE.startswith("http"):
            urllib.request.urlretrieve(f"{SOURCE}/{f}", target)
        else:
            shutil.copy(pathlib.Path(SOURCE) / f, target)
except Exception as e:
    raise SystemExit(f"Could not fetch the course files ({e}). Check the internet connection, then Runtime > Run all again.")
sys.path.insert(0, str(folder.resolve()))
get_ipython().run_line_magic("matplotlib", "inline")
import classroom as lesson
importlib.reload(lesson)
lesson.load(folder)
print("Ready. The course data is loaded.")
print("The forecasts were computed in advance, using only what was known at each forecast date.")


Ready. The course data is loaded.
The forecasts were computed in advance, using only what was known at each forecast date.


## 1. A forecast is not an order

**Lead time** is the wait between placing and receiving an order. We review stock every week; the **protection period** covers lead time plus that review week.

**Safety stock** is a cushion for uncertain demand. **Stock target**, also called the order up to level, is forecast demand over the protection period plus safety stock. **Order now** is the shortfall between that target and stock already on the shelf or on order, assuming no backorders.

This worked example uses round teaching numbers: five forecasts of 180, 190, 210, 200 and 220 units. The forecasts sum to 1,000; the 95 percent planning target adds 148 units of safety stock.

In [2]:
lesson.worked_order()

Teaching scenario: five weekly forecasts, one order decision. These are round illustrative numbers.


,Units
Quantity,
Forecast over five weeks,1000
Safety stock,148
Stock target,1148
Already on the shelf,300
Already on order,500
Order now,348


We assume no backorders and arrivals as scheduled. The stock target is not the quantity to order.


**What to see.** The stock target is 1,148 units, but the order is 348 because 800 units are already on the shelf or on order. Ordering the whole target would count that stock twice.

## 2. Choose a service target

A **cycle service level** is the probability of completing a replenishment cycle without a stockout. A target of 95 percent is a planning assumption, not a promise about the percentage of units served.

**Fill rate** is the share of demand actually served. The replay below reports fill rate, weeks with a stockout and average stock. We do not compare fill rate directly with the cycle service target.

The global model is trained before 2025. Error variability is measured on 25 weeks from January to June; the stock rule is then replayed on 26 weeks from 30 June to 22 December. Orders are rounded up to whole units.

In [3]:
lesson.stock_decision('Plush Bear, store', 95)

Plush Bear, store: cycle service target 95 %; supplier lead time 4 weeks.


,Observed result
Average stock on the shelf,289 units
Fill rate: share of demand served,100.0 %
Weeks with a stockout,0 of 26 weeks
Demand not served,0 units


Safety stock allowance: about 147 units. This is part of the stock target, not the order quantity.
A cycle service target is a planning probability; fill rate measures units served. They are different measures.


**Try it.** Move Target % from 50 to 80, then to 95 and 99. Does every increase serve more demand? Then choose Puzzle 1000 and Vacuum Filter; the same target need not deliver the same observed outcome.

In [4]:
stock_controls = lesson.stock_widget()

The interactive controls appear when you run this notebook in Colab. Until then, use the static example above.


**If the slider is unavailable**, compare these preset settings instead.

In [5]:
lesson.stock_table('Plush Bear, store')

,"Average stock, units",Fill rate,Weeks with a stockout,Units not served
Cycle service target,,,,
50 %,181,98.0 %,7,85
80 %,228,100.0 %,0,0
90 %,261,100.0 %,0,0
95 %,289,100.0 %,0,0
99 %,340,100.0 %,0,0


**What to see.** Compare each increase in stock with the change in fill rate. If fill rate is unchanged, the extra stock served no additional demand in these test weeks; this does not prove that the extra cushion is useless in every future period.

## 3. Compare the products

All three products below use a 95 percent cycle service target. “Weeks of demand on shelf” divides average stock by average demand during the same replay period.

In [6]:
lesson.stock_compare()

,Fill rate at 95 % target,Weeks with a stockout,Weeks of demand on shelf
Product,,,
"Plush Bear, store",100.0 %,0,1.7
"Puzzle 1000, store",87.2 %,6,1.9
"Vacuum Filter, store",93.3 %,1,7.5


**What to see.** A high target can coexist with shortages when the error assumptions are poor. A slow mover can hold many weeks of demand in only a few units; inspect unit cost before calling that stock excessive.

The safety stock calculation assumes independent weekly errors, a stable error distribution and an approximate bell curve. Those assumptions are especially weak for sparse demand. Treat the replay as a check on the rule, not a service guarantee.

## 4. Your management recommendation

Choose one product and complete this in your own notes:

**“For ___, we would trial a cycle service target of ___ percent. The replay held about ___ units and served ___ percent of demand. We accept ___; before using this in a business, we would check ___.”**

Compare your answer with another pair. Different choices can be reasonable when shortage and holding costs differ. You have completed the core exercise; the next sections are optional.

## 5. Optional: what is a better forecast worth?

Keep the same 95 percent target and change the forecasting approach. Each method uses its own error estimate. Read stock and service together; lower stock with substantially lower service is a different policy, not automatically a better one.

In [7]:
lesson.forecast_value()

Plush Bear, store. Same 95 % target, each method uses its own measured errors.
Unit cost EUR 13.83; annual holding rate 20%.
Annualised stock holding cost only: a scenario based on the replay average, excluding shortages and implementation.


,"Average stock, units",Fill rate,Annualised holding cost
Method,,,
Repeat last year,351,100.0 %,EUR 971
Smooth recent demand,208,88.0 %,EUR 574
Global model,289,100.0 %,EUR 798


**What to see.** The last column prices the average stock using this product’s unit cost and a 20 percent annual holding rate. It is an annualised scenario based on a half-year replay, not realised annual savings or a complete investment case; shortages and implementation costs are excluded.

## 6. Optional: one Christmas order

The **newsvendor** problem is one order for one selling period. A unit short loses margin; a leftover unit costs its purchase price minus what we recover afterward.

The **critical ratio** is shortage cost divided by shortage cost plus leftover cost. It chooses a **quantile**: a demand level with a specified probability of not being exceeded. Here demand is approximated with a bell curve, so the 50th percentile is its mean.

This is a separate decision assuming the order can arrive for the selling week; it does not use the replenishment lead time. Use recovery below 100 percent: with no leftover cost, this unbounded model has no finite optimal order.

In [8]:
lesson.christmas_display(50)

Puzzle 1000, store, 15 December 2025: forecast 88 units; actual 41.
Recover 50 % of purchase cost on leftovers: a unit short costs EUR 23.23, a leftover costs EUR 19.43.
Critical ratio 0.54: choose that quantile of demand. Suggested order 92 units.
A quantile is a demand level with a chosen probability of not being exceeded. Here we approximate demand with a bell curve.
This separate exercise assumes one order can arrive for the selling week; it does not simulate the supplier lead time.


In [9]:
christmas_controls = lesson.christmas_widget()

The interactive controls appear when you run this notebook in Colab. Until then, use the static example above.


**What to see.** Higher recovery makes leftovers cheaper and increases the order, while the forecast stays fixed. This shows why a forecast alone cannot choose an order quantity.

**Take home.** Keep the forecast, its error and the business cost of a mistake together. Your recommendation should state a choice, evidence and a limitation.